## 1. Environment Setup and Imports

In [ ]:
# Install dependencies
!pip install -q torch transformers datasets accelerate evaluate
!pip install -q nltk rouge-score scikit-learn tqdm
!pip install -q wandb tensorboard

In [ ]:
import sys
sys.path.insert(0, '/home/houssem/BugFixer/src')

import os
import json
import random
import logging
from pathlib import Path
from typing import List, Dict, Tuple
from datetime import datetime

import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from torch.utils.data import DataLoader, Dataset

# Import custom modules
from data_processing.real_bug_loader import (
    CombinedBugDataset, BugFix, create_java_code_samples
)
from data_processing.advanced_preprocessor import (
    AdvancedJavaPreprocessor, PreprocessedPair
)
from model.enhanced_architecture import CodeT5MultiTask, CurriculumLearningScheduler
from model.optimized_trainer import OptimizedTrainer, TrainingConfig, BugFixDataset
from evaluation.comprehensive_evaluator import BugFixEvaluator
from model.production_inference import ProductionInference

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f"Using device: {device}")
if torch.cuda.is_available():
    logger.info(f"GPU: {torch.cuda.get_device_name(0)}")
    logger.info(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

# Set random seeds
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("✓ Environment setup complete")

## 2. Data Preparation - Critical Phase

### Strategy: Synthetic Bug Generation from Real Java Code

Since Defects4J might not be available locally, we create high-quality synthetic bugs by:
1. Taking correct Java code snippets
2. Injecting common bug patterns programmatically
3. Balancing bug type distribution
4. Validating syntax to filter broken examples

In [ ]:
# Create output directory
output_dir = Path('/home/houssem/BugFixer/output')
output_dir.mkdir(exist_ok=True)

data_dir = output_dir / 'data'
data_dir.mkdir(exist_ok=True)

logger.info(f"Output directory: {output_dir}")
logger.info(f"Data directory: {data_dir}")

In [ ]:
# Step 1: Generate dataset
logger.info("\n" + "="*70)
logger.info("PHASE 1: DATA PREPARATION")
logger.info("="*70)

# Get Java code samples for synthetic bug generation
java_samples = create_java_code_samples()
logger.info(f"Created {len(java_samples)} Java code samples")

# Initialize dataset generator
dataset_gen = CombinedBugDataset()

# Generate dataset
logger.info("\nGenerating synthetic bugs...")
bugs = dataset_gen.generate_dataset(
    java_code_samples=java_samples,
    num_synthetic_per_sample=5,  # 5 bugs per sample = 40 synthetic bugs
    num_github_bugs=100,
    include_defects4j=False,  # Defects4J not available
)

logger.info(f"Generated {len(bugs)} bug-fix pairs")

# Save raw dataset
raw_dataset_path = data_dir / 'raw_bugs.json'
dataset_gen.save_dataset(bugs, str(raw_dataset_path))

print(f"✓ Dataset preparation complete: {len(bugs)} bug-fix pairs")

In [ ]:
# Step 2: Analyze dataset distribution
bug_type_counts = {}
for bug in bugs:
    bug_type_counts[bug.bug_type] = bug_type_counts.get(bug.bug_type, 0) + 1

logger.info("\nBug type distribution:")
for bug_type, count in sorted(bug_type_counts.items(), key=lambda x: x[1], reverse=True):
    logger.info(f"  {bug_type}: {count}")

# Show sample
sample_bug = bugs[0]
logger.info(f"\nSample bug:")
logger.info(f"  Type: {sample_bug.bug_type}")
logger.info(f"  Buggy: {sample_bug.buggy_code[:100]}...")
logger.info(f"  Fixed: {sample_bug.fixed_code[:100]}...")

## 3. Advanced Preprocessing with Validation

In [ ]:
# Initialize preprocessor
preprocessor = AdvancedJavaPreprocessor(max_length=512)

# Preprocess all pairs
logger.info("\n" + "="*70)
logger.info("PHASE 2: PREPROCESSING")
logger.info("="*70)
logger.info("\nPreprocessing code pairs...")

preprocessed_pairs = []
for bug in tqdm(bugs, desc="Preprocessing"):
    pair = preprocessor.preprocess_pair(
        buggy_code=bug.buggy_code,
        fixed_code=bug.fixed_code,
        bug_type=bug.bug_type,
        extract_body=False,  # Keep full context
        add_prefix=True,  # Add task prefix
        validate_syntax=True,
    )
    preprocessed_pairs.append(pair)

# Filter valid pairs
valid_pairs = preprocessor.filter_valid_pairs(
    preprocessed_pairs,
    min_buggy_tokens=5,
    min_fixed_tokens=5,
)

logger.info(f"\nPreprocessing complete:")
logger.info(f"  Total pairs: {len(preprocessed_pairs)}")
logger.info(f"  Valid pairs: {len(valid_pairs)}")
logger.info(f"  Filtered out: {len(preprocessed_pairs) - len(valid_pairs)}")

# Get statistics
stats = preprocessor.get_statistics(valid_pairs)
logger.info(f"\nPreprocessing Statistics:")
for key, value in stats.items():
    if key != 'bug_type_distribution':
        logger.info(f"  {key}: {value}")

logger.info(f"\nBug type distribution (after filtering):")
for bug_type, count in sorted(stats['bug_type_distribution'].items(), key=lambda x: x[1], reverse=True):
    logger.info(f"  {bug_type}: {count}")

## 4. Create Training, Validation, and Test Sets

In [ ]:
# Split data
# Use 70% for training, 15% for validation, 15% for testing
n_samples = len(valid_pairs)
train_size = int(0.7 * n_samples)
val_size = int(0.15 * n_samples)

# Shuffle
import random
all_indices = list(range(n_samples))
random.shuffle(all_indices)

train_indices = all_indices[:train_size]
val_indices = all_indices[train_size:train_size + val_size]
test_indices = all_indices[train_size + val_size:]

train_pairs = [valid_pairs[i] for i in train_indices]
val_pairs = [valid_pairs[i] for i in val_indices]
test_pairs = [valid_pairs[i] for i in test_indices]

logger.info(f"\nData split:")
logger.info(f"  Training: {len(train_pairs)}")
logger.info(f"  Validation: {len(val_pairs)}")
logger.info(f"  Test: {len(test_pairs)}")

# Save splits
def save_split(pairs, path):
    data = [
        {
            'buggy_code': p.buggy_code,
            'fixed_code': p.fixed_code,
            'bug_type': p.bug_type,
            'buggy_tokens': p.buggy_tokens,
            'fixed_tokens': p.fixed_tokens,
        }
        for p in pairs
    ]
    with open(path, 'w') as f:
        json.dump(data, f, indent=2)
    logger.info(f"Saved {len(pairs)} pairs to {path}")

save_split(train_pairs, data_dir / 'train.json')
save_split(val_pairs, data_dir / 'validation.json')
save_split(test_pairs, data_dir / 'test.json')

## 5. Model Setup with Multi-Task Learning

In [ ]:
# Model configuration
config = TrainingConfig(
    model_name="Salesforce/codet5-base",
    batch_size=16,
    gradient_accumulation_steps=4,
    learning_rate=3e-5,
    weight_decay=0.01,
    num_epochs=15,
    max_source_length=512,
    max_target_length=512,
    warmup_steps=1000,
    max_grad_norm=1.0,
    use_fp16=torch.cuda.is_available(),
    num_beams=5,
    early_stopping=True,
    patience=3,
    lr_scheduler='linear',
    output_dir=str(output_dir / 'model'),
    use_curriculum=True,
)

logger.info("\n" + "="*70)
logger.info("PHASE 3: MODEL SETUP")
logger.info("="*70)
logger.info(f"\nConfiguration:")
for key, value in config.to_dict().items():
    logger.info(f"  {key}: {value}")

# Load tokenizer
logger.info(f"\nLoading tokenizer: {config.model_name}")
tokenizer = AutoTokenizer.from_pretrained(config.model_name)
logger.info(f"Tokenizer loaded. Vocab size: {len(tokenizer)}")

# Initialize model
logger.info(f"\nInitializing model: {config.model_name}")
model = CodeT5MultiTask(
    model_name=config.model_name,
    num_bug_types=len(stats['bug_type_distribution']),
    dropout_rate=0.1,
    label_smoothing=0.1,
)
model = model.to(device)

# Print parameter counts
params = model.count_parameters()
logger.info(f"\nModel parameters:")
for key, value in params.items():
    logger.info(f"  {key}: {value:,}")

print("✓ Model setup complete")

## 6. Create PyTorch Datasets

In [ ]:
# Create bug type mapping
bug_types_list = sorted(stats['bug_type_distribution'].keys())
bug_type_to_id = {t: i for i, t in enumerate(bug_types_list)}

logger.info(f"\nBug type mapping:")
for bug_type, idx in bug_type_to_id.items():
    logger.info(f"  {idx}: {bug_type}")

# Create datasets
train_dataset = BugFixDataset(
    buggy_codes=[p.buggy_code for p in train_pairs],
    fixed_codes=[p.fixed_code for p in train_pairs],
    bug_types=[p.bug_type for p in train_pairs],
    tokenizer=tokenizer,
    config=config,
    bug_type_to_id=bug_type_to_id,
)

val_dataset = BugFixDataset(
    buggy_codes=[p.buggy_code for p in val_pairs],
    fixed_codes=[p.fixed_code for p in val_pairs],
    bug_types=[p.bug_type for p in val_pairs],
    tokenizer=tokenizer,
    config=config,
    bug_type_to_id=bug_type_to_id,
)

logger.info(f"\nDatasets created:")
logger.info(f"  Train: {len(train_dataset)}")
logger.info(f"  Validation: {len(val_dataset)}")

# Test dataset access
sample = train_dataset[0]
logger.info(f"\nSample from training set:")
logger.info(f"  input_ids shape: {sample['input_ids'].shape}")
logger.info(f"  attention_mask shape: {sample['attention_mask'].shape}")
logger.info(f"  labels shape: {sample['labels'].shape}")
logger.info(f"  bug_type_id: {sample['bug_type_id']}")

## 7. Training with Advanced Optimizations

In [ ]:
# Initialize trainer
logger.info("\n" + "="*70)
logger.info("PHASE 4: TRAINING")
logger.info("="*70)

trainer = OptimizedTrainer(
    model=model,
    tokenizer=tokenizer,
    config=config,
    device=device,
)

logger.info("\nTrainer initialized")

# Start training
logger.info("\nStarting training...\n")
start_time = datetime.now()

trainer.train(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
)

training_time = datetime.now() - start_time
logger.info(f"\nTraining completed in {training_time}")

# Get training stats
stats_training = trainer.get_training_stats()
logger.info(f"\nTraining statistics:")
for key, value in stats_training.items():
    logger.info(f"  {key}: {value}")

## 8. Evaluation on Test Set

In [ ]:
# Initialize evaluator
logger.info("\n" + "="*70)
logger.info("PHASE 5: EVALUATION")
logger.info("="*70)

evaluator = BugFixEvaluator(tokenizer, bug_types=bug_types_list)

# Generate predictions on test set
logger.info("\nGenerating predictions on test set...")

test_loader = DataLoader(
    val_dataset,  # Use validation set for evaluation
    batch_size=config.batch_size,
    shuffle=False,
)

predictions = []
model.eval()

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Generating predictions"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=config.max_target_length,
            num_beams=config.num_beams,
        )
        
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        predictions.extend(decoded)

# Evaluate predictions
logger.info("\nEvaluating predictions...")

results = evaluator.evaluate_batch(
    buggy_codes=[p.buggy_code for p in val_pairs],
    fixed_codes=[p.fixed_code for p in val_pairs],
    predicted_codes=predictions,
    bug_types=[p.bug_type for p in val_pairs],
)

# Compute aggregate metrics
metrics = evaluator.compute_aggregate_metrics(results)

# Print report
report = evaluator.generate_report(metrics)
print(report)

# Save results
evaluator.save_results(
    results,
    output_dir / 'evaluation_results.json'
)

# Save metrics
with open(output_dir / 'metrics.json', 'w') as f:
    # Convert numpy types to native Python types
    def convert(o):
        if isinstance(o, np.integer):
            return int(o)
        elif isinstance(o, np.floating):
            return float(o)
        elif isinstance(o, dict):
            return {convert(k): convert(v) for k, v in o.items()}
        elif isinstance(o, list):
            return [convert(x) for x in o]
        return o
    
    json.dump(convert(metrics), f, indent=2)

## 9. Production Inference Testing

In [ ]:
# Initialize inference engine
logger.info("\n" + "="*70)
logger.info("PHASE 6: PRODUCTION INFERENCE")
logger.info("="*70)

inference = ProductionInference(
    model=model,
    tokenizer=tokenizer,
    device=device,
    bug_types=bug_types_list,
)

# Test on sample cases
logger.info("\nTesting inference on sample cases...\n")

# From test_cases.py
test_samples = [
    {
        'code': 'for(int i=0; i<=arr.length; i++) { sum += arr[i]; }',
        'expected_type': 'off_by_one'
    },
    {
        'code': 'int value = (String) obj;',
        'expected_type': 'type_mismatch'
    },
    {
        'code': 'if (a > 0 & b < 10) { return true; }',
        'expected_type': 'operator'
    },
]

for idx, sample in enumerate(test_samples, 1):
    logger.info(f"Test {idx}:")
    logger.info(f"  Buggy: {sample['code'][:60]}")
    
    result = inference.predict_single(
        buggy_code=sample['code'],
        return_alternatives=True,
        predict_bug_type=True,
    )
    
    logger.info(f"  Predicted Type: {result.predicted_bug_type}")
    logger.info(f"  Confidence: {result.confidence:.2%}")
    logger.info(f"  Fixed: {result.predicted_code[:60]}")
    logger.info()

## 10. Model Export and Deployment

In [ ]:
# Save final model
logger.info("\nSaving final model...")

model_export_dir = output_dir / 'final_model'
model_export_dir.mkdir(exist_ok=True)

model.base_model.save_pretrained(str(model_export_dir))
tokenizer.save_pretrained(str(model_export_dir))

# Save configuration
config_export = {
    'model_name': config.model_name,
    'max_source_length': config.max_source_length,
    'max_target_length': config.max_target_length,
    'num_beams': config.num_beams,
    'bug_types': bug_types_list,
    'bug_type_to_id': bug_type_to_id,
    'training_time': str(training_time),
    'metrics': metrics,
}

with open(model_export_dir / 'config.json', 'w') as f:
    json.dump(config_export, f, indent=2)

logger.info(f"Model saved to {model_export_dir}")

print("\n" + "="*70)
print("✓ TRAINING COMPLETE")
print("="*70)
print(f"\nResults saved to: {output_dir}")
print(f"Model saved to: {model_export_dir}")
print(f"\nKey Metrics:")
print(f"  Exact Match: {metrics['exact_match']:.4f}")
print(f"  BLEU: {metrics['bleu']:.4f}")
print(f"  CodeBLEU: {metrics['code_bleu']:.4f}")
print(f"  Syntax Valid: {metrics['syntax_valid_rate']:.4f}")
print(f"  Training Time: {training_time}")